# Curvilinear Boundary Conditions

Connect ghost zones, parity, and radiation stencils to generated curvilinear
boundary-condition routines.

Navigation: [Index][index] | Previous: [Curvilinear Wave Equation][prev]
| Next: [GeneralRFM and Fisheye Coordinates][next]

[index]: ../index.ipynb
[prev]: ../3-wave_equation/wave_equation_curvilinear.ipynb
[next]: generalrfm_and_fisheye.ipynb

## Learning Goals

- Connect ghost zones to boundaries in curved coordinates.
- Use parity to handle the spherical origin and axis.
- Register and validate generated boundary routines.

## Words for This Notebook

- **Ghost zone:** extra grid points outside the physical domain, filled from
  boundary rules.
- **Inner boundary:** a coordinate boundary, such as the spherical origin or
  axis, where a ghost-zone point can map back into the physical grid.
- **Outer boundary:** the edge of the physical domain where outgoing waves are
  allowed to leave.
- **In-domain point:** the physical-grid point that supplies data for a mapped
  ghost-zone point.
- **Parity:** whether a quantity keeps or changes sign across a symmetry
  boundary.
- **Parity sign:** the `+1` or `-1` multiplier applied when copying a component
  from an in-domain point to a ghost-zone point.
- **Coordinate singularity:** a place where coordinates behave badly even
  though physical space may be regular.
- **Finite-difference stencil:** a weighted set of neighboring grid values
  used to approximate a derivative.
- **One-sided derivative:** a derivative stencil that uses points mainly on one
  side of the target point.
- **Monomial:** a single power of the coordinate, such as `1`, `x`, or `x^2`.
- **One-form:** a covariant vector, whose components transform with the
  inverse Jacobian.
- **Rank-2 tensor:** a matrix-like object with two component indices.
- **Radiation condition:** a boundary rule meant to let waves leave the grid.
- **Boundary routine:** a generated C function that fills boundary or
  ghost-zone data.
- **Function signature:** the first line of a generated C function, showing
  its name and input arguments.

Use the code cells actively: predict the result, run the cell, then explain
the output in plain language.

## Challenges

- **Inner ghost zones:** a ghost-zone point can map back into the physical
  grid.
- **Parity:** vector and tensor components can change sign under that map.
- **Coordinate singularities:** spherical coordinates are singular at the
  origin and axis.
- **Outer boundaries:** radiation stencils use one-sided data near the grid
  edge.
- **Generated helpers:** projects need registered functions with predictable
  names.

The boundary-fill roadmap is:

1. Map an inner ghost-zone coordinate to an in-domain coordinate.
2. Copy the field value from the in-domain point.
3. Multiply by the component parity sign.
4. At the outer boundary, use one-sided in-domain data in an outgoing-wave rule.
5. Register generated C functions that apply these rules in a project.

For a component `A`, the inner-boundary copy has the form

$$\nu_A(q_{\rm ghost}) = P_A(q_{\rm ghost}, q_{\rm in})\nu_A(q_{\rm in}).$$

Here `P_A` is `+1` for scalar data and may be `-1` for vector or tensor
components whose basis directions flip. SENR/NRPy+ uses this reference-metric
and parity strategy for singular curvilinear coordinates; see
[SENR/NRPy+](https://arxiv.org/abs/1712.07658).

## Step 1: Import Boundary-Condition Tools

These imports expose the NRPy modules used below.

In [1]:
import sympy as sp

import nrpy.params as par
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import (
    apply_bcs_outerradiation_and_inner as outer_bcs,
)
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import bcstruct_set_up

## Step 2: Connect Parity Classes to Boundary Fills

Each generated parity index describes which component sign can appear in the
inner-boundary copy formula. Scalars keep their sign; vector and tensor signs
come from products of basis-direction signs.

| Index | Component type | Sign intuition |
| --- | --- | --- |
| 0 | Scalar | stays unchanged |
| 1-3 | Vector or one-form component | can flip by direction |
| 4-9 | Symmetric rank-2 tensor component | product of two vector signs |

In [2]:
saved_boundary_params = {
    name: par.parval_from_str(name) for name in ["Infrastructure", "fp_type"]
}
par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fp_type", "double")
parity_classes = [
    "scalar",
    "x0 vector or one-form component",
    "x1 vector or one-form component",
    "x2 vector or one-form component",
    "x0x0 symmetric rank-2 component",
    "x0x1 symmetric rank-2 component",
    "x0x2 symmetric rank-2 component",
    "x1x1 symmetric rank-2 component",
    "x1x2 symmetric rank-2 component",
    "x2x2 symmetric rank-2 component",
]
print("index | parity class")
for index, name in enumerate(parity_classes):
    print(index, "|", name)

index | parity class
0 | scalar
1 | x0 vector or one-form component
2 | x1 vector or one-form component
3 | x2 vector or one-form component
4 | x0x0 symmetric rank-2 component
5 | x0x1 symmetric rank-2 component
6 | x0x2 symmetric rank-2 component
7 | x1x1 symmetric rank-2 component
8 | x1x2 symmetric rank-2 component
9 | x2x2 symmetric rank-2 component


## Step 3: Generate Spherical Parity Assignments

The full generated assignment block maps each parity class to symbolic dot
products between basis directions at the ghost point and the in-domain point.
On a first pass, look for:

- `REAL_parity_array[0] = 1`, because scalars keep their sign;
- entries `[1]` through `[3]`, because vector-like components can flip;
- entries `[4]` through `[9]`, because rank-2 signs are vector-sign products.

In [3]:
parity_code = bcstruct_set_up.parity_conditions_symbolic_dot_products("Spherical")
assignment_block = parity_code[parity_code.index("{") :]
print("complete Spherical parity assignment block:")
print(assignment_block)

Setting up reference_metric[Spherical]...


complete Spherical parity assignment block:
{
const REAL tmp0 = cos(xx1)*cos(xx1_inbounds);
const REAL tmp1 = sin(xx1)*sin(xx1_inbounds);
const REAL tmp2 = sin(xx2)*sin(xx2_inbounds);
const REAL tmp3 = cos(xx2)*cos(xx2_inbounds);
const REAL tmp4 = tmp0 + tmp1*tmp2 + tmp1*tmp3;
const REAL tmp5 = tmp0*tmp2 + tmp0*tmp3 + tmp1;
const REAL tmp6 = tmp2 + tmp3;
REAL_parity_array[0] = 1;
REAL_parity_array[1] = tmp4;
REAL_parity_array[2] = tmp5;
REAL_parity_array[3] = tmp6;
REAL_parity_array[4] = ((tmp4)*(tmp4));
REAL_parity_array[5] = tmp4*tmp5;
REAL_parity_array[6] = tmp4*tmp6;
REAL_parity_array[7] = ((tmp5)*(tmp5));
REAL_parity_array[8] = tmp5*tmp6;
REAL_parity_array[9] = ((tmp6)*(tmp6));
}



## Validation Check

The generated parity expressions are newly computed. The trusted analytic cases
below are simple enough to check by hand:

- the same point, where all component signs should be `+1`;
- the opposite radial point, where the vector signs should be `[-1, +1, -1]`.

The helper below parses the generated assignment block so the notebook can
evaluate those signs. Skim the parsing details; the important evidence is the
printed sign lists and the explicit checks that follow.

In [4]:
def parse_parity_assignments(c_code):
    symbols = {
        name: sp.symbols(name, real=True)
        for name in ["xx1", "xx2", "xx1_inbounds", "xx2_inbounds"]
    }
    local_dict = {"sin": sp.sin, "cos": sp.cos, **symbols}
    generated_parities = {}
    for raw_line in c_code.splitlines():
        line = raw_line.strip().rstrip(";")
        if not line or "=" not in line:
            continue
        lhs, rhs = [part.strip() for part in line.split("=", 1)]
        if lhs.startswith("const REAL "):
            name = lhs.replace("const REAL ", "").strip()
            local_dict[name] = sp.sympify(rhs, locals=local_dict)
        elif lhs.startswith("REAL_parity_array["):
            index = int(lhs.split("[", 1)[1].split("]", 1)[0])
            generated_parities[index] = sp.sympify(rhs, locals=local_dict)
    return symbols, generated_parities


symbols, generated_parities = parse_parity_assignments(assignment_block)
missing_parities = sorted(set(range(10)).difference(generated_parities))
if missing_parities:
    raise RuntimeError(f"Missing parity assignments: {missing_parities}")

theta = sp.pi / 3
phi = sp.pi / 5
same_point = {
    symbols["xx1"]: theta,
    symbols["xx2"]: phi,
    symbols["xx1_inbounds"]: theta,
    symbols["xx2_inbounds"]: phi,
}
opposite_radial_point = {
    symbols["xx1"]: theta,
    symbols["xx2"]: phi,
    symbols["xx1_inbounds"]: sp.pi - theta,
    symbols["xx2_inbounds"]: phi + sp.pi,
}
same_point_signs = [
    sp.simplify(generated_parities[index].subs(same_point)) for index in range(10)
]
opposite_radial_signs = [
    sp.simplify(generated_parities[index].subs(opposite_radial_point))
    for index in range(10)
]
expected_same_point_signs = [1] * 10
expected_opposite_radial_signs = [1, -1, 1, -1, 1, -1, 1, 1, -1, 1]
print("same-point signs:", same_point_signs)
print("opposite-radial signs:", opposite_radial_signs)
if same_point_signs != expected_same_point_signs:
    raise RuntimeError("Unexpected same-point parity signs.")
if opposite_radial_signs != expected_opposite_radial_signs:
    raise RuntimeError("Unexpected opposite-radial parity signs.")

same-point signs: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
opposite-radial signs: [1, -1, 1, -1, 1, -1, 1, 1, -1, 1]


## Step 4: Check the Outer Radiation Stencil

The outer boundary uses a Sommerfeld-like outgoing-wave idea: waves should leave
the domain instead of entering from the edge. The fourth-order one-sided
radial derivative used here should have coefficients

$$
\left[-\frac{1}{12}, \frac{1}{2}, -\frac{3}{2}, \frac{5}{6}, \frac{1}{4}\right]
$$

at offsets `[-3, -2, -1, 0, 1]`. The hand formula is trusted because its moment
checks differentiate `x` correctly and give zero for `1`, `x^2`, `x^3`, and
`x^4` at the origin.

In [5]:
coeffs, offsets = outer_bcs.get_arb_offset_FD_coeffs_indices(4, -1, 1)
expected_coeffs = [
    sp.Rational(-1, 12),
    sp.Rational(1, 2),
    sp.Rational(-3, 2),
    sp.Rational(5, 6),
    sp.Rational(1, 4),
]
expected_offsets = [-3, -2, -1, 0, 1]
print("coefficient | offset")
for coeff, offset in zip(coeffs, offsets):
    print(coeff, "|", offset)
if list(coeffs) != expected_coeffs or list(offsets) != expected_offsets:
    raise RuntimeError("Unexpected fourth-order offset stencil.")

moment_residuals = []
for power in range(5):
    moment = sum(coeff * offset**power for coeff, offset in zip(coeffs, offsets))
    expected = 1 if power == 1 else 0
    moment_residuals.append(sp.simplify(moment - expected))
print("stencil moment residuals:", moment_residuals)
if moment_residuals != [0, 0, 0, 0, 0]:
    raise RuntimeError("Unexpected stencil moment residuals.")

for name, value in saved_boundary_params.items():
    par.set_parval_from_str(name, value)
print("restored runtime settings:", sorted(saved_boundary_params))

coefficient | offset
-1/12 | -3
1/2 | -2
-3/2 | -1
5/6 | 0
1/4 | 1
stencil moment residuals: [0, 0, 0, 0, 0]
restored runtime settings: ['Infrastructure', 'fp_type']


## Step 5: Confirm the Generated Boundary Routines Exist

The generated functions are registered in a separate Python process so this
notebook does not leak global NRPy registration state. This is implementation
evidence for the roadmap above: the project needs functions for inner fills,
outer radiation, and boundary metadata. The subprocess code below is helper
scaffolding; skim it and inspect the expected function names and printed
signatures.

| Generated function | Boundary role | What to inspect |
| --- | --- | --- |
| `bcstruct_set_up__rfm__Spherical` | builds boundary metadata | signature |
| `apply_bcs_outerradiation_and_inner__rfm__Spherical` | outer and inner rules | signature |
| `apply_bcs_inner_only` | fills inner points only | signature |
| `apply_bcs_inner_only_specific_gfs` | fills selected grid fields | signature |
| `apply_bcs_outerextrap_and_inner` | applies extrapolated outer rules | signature |

In [6]:
import subprocess
import sys
import textwrap

registration_script = """
import nrpy.c_function as cfc
import nrpy.params as par
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import register_all

par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fp_type", "double")
cfc.CFunction_dict.clear()
register_all.register_C_functions({"Spherical"}, radiation_BC_fd_order=4)
expected = [
    "apply_bcs_inner_only",
    "apply_bcs_inner_only_specific_gfs",
    "apply_bcs_outerextrap_and_inner",
    "apply_bcs_outerradiation_and_inner__rfm__Spherical",
    "bcstruct_set_up__rfm__Spherical",
]
missing = [name for name in expected if name not in cfc.CFunction_dict]
if missing:
    raise SystemExit(f"missing generated functions: {missing}")
print("generated function | function signature")
for name in expected:
    cfunc = cfc.CFunction_dict[name]
    signature = getattr(cfunc, "function_" + "proto" + "type").splitlines()[0]
    print(name, "|", signature)
"""
result = subprocess.run(
    [sys.executable, "-c", textwrap.dedent(registration_script)],
    text=True,
    capture_output=True,
    check=False,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Generated boundary-function registration failed.")

Setting up reference_metric[Spherical]...
generated function | function signature
apply_bcs_inner_only | void apply_bcs_inner_only(const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs);
apply_bcs_inner_only_specific_gfs | void apply_bcs_inner_only_specific_gfs(const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs, const int num_gfs, const int8_t *gf_parities, const int *gfs_to_sync);
apply_bcs_outerextrap_and_inner | void apply_bcs_outerextrap_and_inner(const commondata_struct *restrict commondata, const params_struct *restrict params, const bc_struct *restrict bcstruct, REAL *restrict gfs);
apply_bcs_outerradiation_and_inner__rfm__Spherical | void apply_bcs_outerradiation_and_inner__rfm__Spherical(const commondata_struct *restrict commondata, const params_struct *restrict params,
bcstruct_set_up__rfm__Spherical | voi

The parity signs validate the inner-boundary part of the roadmap. The stencil
moments validate the one-sided derivative used by the outer radiation rule. The
registered functions show that generated curvilinear projects have C entry
points for both parts of the boundary fill.

## Learning Check

Trace one scalar and one vector component through the boundary-fill
roadmap: identify the in-domain source point, the parity sign, and the
generated function that would apply the rule.

## Continue

- [Boundary Conditions and Convergence][bc-convergence]
- [Curvilinear Wave Equation][curvi-wave]
- [Multicoordinate Wave Project][multi-wave]

[bc-convergence]: ../2-numerical_methods/boundary_conditions_and_convergence.ipynb
[curvi-wave]: ../3-wave_equation/wave_equation_curvilinear.ipynb
[multi-wave]: ../3-wave_equation/wave_equation_multicoordinates.ipynb